# Object Detection using TAO DINO

Transfer learning is the process of transferring learned features from one application to another. It is a commonly used training technique where you use a model trained on one task and re-train to use it on a different task. 

Train Adapt Optimize (TAO) Toolkit  is a simple and easy-to-use Python based AI toolkit for taking purpose-built AI models and customizing them with users' own data.

<img align="center" src="https://d29g4g2dyqv443.cloudfront.net/sites/default/files/akamai/TAO/tlt-tao-toolkit-bring-your-own-model-diagram.png" width="1080">

## What is DINO?

[DINO](https://arxiv.org/abs/2203.03605) is a state of the art transformer-based object detection model. Similar to Deformable DETR, DINO does not use heuristics based methods like NMS or IOU assignment found in convolution-based object detection models like Faster RCNN. Compared to Deformable DETR, DINO uses de-noising during training which can help training to converge faster.

In TAO, three different types of backbone networks are supported: [ResNet50](https://arxiv.org/abs/1512.03385), [GCViT](https://arxiv.org/abs/2206.09959), and [FAN](https://arxiv.org/abs/2204.12451). In this notebook, we use the most advanced network called FAN which is also a transformer-based classification network. For more details about training FAN backbones, please refer to the classification pytorch notebook.

### Sample prediction of FAN-Tiny + DINO model
<img align="center" src="sample.jpg" width="960">

## Learning Objectives

In this notebook, you will learn how to leverage the simplicity and convenience of TAO to:

* Take a pretrained model and train an DINO model on COCO dataset
* Evaluate the trained model
* Run inference with the trained model and visualize the result
* Export the trained model to a .onnx file for deployment to DeepStream
* Generate TensorRT engine using tao-deploy and verify the engine through evaluation

At the end of this notebook, you will have generated a trained `dino` model
which you may deploy via [DeepStream](https://developer.nvidia.com/deepstream-sdk).

## Table of Contents

This notebook shows an example usecase of DINO using Train Adapt Optimize (TAO) Toolkit.

0. [Set up env variables and map drives](#head-0)
1. [Installing the TAO launcher](#head-1)
2. [Prepare dataset and pre-trained model](#head-2)
3. [Provide training specification](#head-3)
4. [Run TAO training](#head-4)
5. [Evaluate a trained model](#head-5)
6. [Visualize inferences](#head-6)
7. [Deploy](#head-7)

## 0. Set up env variables and map drives <a class="anchor" id="head-0"></a>

The following notebook requires the user to set an env variable called the `$LOCAL_PROJECT_DIR` as the path to the users workspace. Please note that the dataset to run this notebook is expected to reside in the `$LOCAL_PROJECT_DIR/data`, while the TAO experiment generated collaterals will be output to `$LOCAL_PROJECT_DIR/dino/results`. More information on how to set up the dataset and the supported steps in the TAO workflow are provided in the subsequent cells.

The TAO launcher uses docker containers under the hood, and **for our data and results directory to be visible to the docker, they need to be mapped**. The launcher can be configured using the config file `~/.tao_mounts.json`. Apart from the mounts, you can also configure additional options like the Environment Variables and amount of Shared Memory available to the TAO launcher. <br>

`IMPORTANT NOTE:` The code below creates a sample `~/.tao_mounts.json`  file. Here, we can map directories in which we save the data, specs, results and cache. You should configure it for your specific case so these directories are correctly visible to the docker container.


In [18]:
import json
import os

# Define the correct source path
source_path = "/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino"

# Check if the source path exists
if not os.path.exists(source_path):
    raise ValueError(f"Mount point source path doesn't exist: {source_path}")

# Define the mounts configuration
tao_configs = {
    "Mounts": [
        {
            "source": "/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit",
            "destination": "/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit",
            "type": "bind",
            "access": "rw"
        },
        {
            "source": "/var/run/docker.sock",
            "destination": "/var/run/docker.sock"
            
        },
        
        
        {
            "source": "/var/host-run",
            "destination": "/var/host-run"
        },
     
    ],
    "DockerOptions": {
        "shm_size": "16G",
        "ulimits": {
            "memlock": -1,
            "stack": 67108864
        },
        "user": "1000:1000",
        "network": "host",
        "privileged": True
    }
}

# Path to the mounts file
mounts_file = os.path.expanduser("~/.tao_mounts.json")

# Writing the mounts file
with open(mounts_file, "w") as mfile:
    json.dump(tao_configs, mfile, indent=4)

# Verify the contents of the mounts file
with open(mounts_file, "r") as mfile:
    print(mfile.read())

{
    "Mounts": [
        {
            "source": "/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit",
            "destination": "/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit",
            "type": "bind",
            "access": "rw"
        },
        {
            "source": "/var/run/docker.sock",
            "destination": "/var/run/docker.sock"
        },
        {
            "source": "/var/host-run",
            "destination": "/var/host-run"
        }
    ],
    "DockerOptions": {
        "shm_size": "16G",
        "ulimits": {
            "memlock": -1,
            "stack": 67108864
        },
        "user": "1000:1000",
        "network": "host",
        "privileged": true
    }
}


In [19]:
import os

# Set the environment variables
os.environ['HOST_DATA_DIR'] = '/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data'
os.environ['HOST_SPECS_DIR'] = '/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/specs'
os.environ['HOST_RESULTS_DIR'] = '/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/results'

os.environ['SPECS_DIR'] = '/workspace/specs'
os.environ['RESULTS_DIR'] = '/workspace/results'

# Create directories if they do not exist
os.makedirs(os.environ['HOST_DATA_DIR'], exist_ok=True)
os.makedirs(os.environ['HOST_SPECS_DIR'], exist_ok=True)
os.makedirs(os.environ['HOST_RESULTS_DIR'], exist_ok=True)

# Verify the environment variables
print(f"HOST_DATA_DIR: {os.environ['HOST_DATA_DIR']}")
print(f"HOST_SPECS_DIR: {os.environ['HOST_SPECS_DIR']}")
print(f"HOST_RESULTS_DIR: {os.environ['HOST_RESULTS_DIR']}")

print(f"SPECS_DIR: {os.environ['SPECS_DIR']}")
print(f"RESULTS_DIR: {os.environ['RESULTS_DIR']}")

# Add the directory to the PATH environment variable
os.environ['PATH'] = os.environ['PATH'] + ':/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/bin'

# Verify the updated PATH
print(f"Updated PATH: {os.environ['PATH']}")

# Verification of directories and files
!ls -l $HOST_DATA_DIR/
!ls -l $HOST_SPECS_DIR/

HOST_DATA_DIR: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data
HOST_SPECS_DIR: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/specs
HOST_RESULTS_DIR: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/results
SPECS_DIR: /workspace/specs
RESULTS_DIR: /workspace/results
Updated PATH: /bin:/home/workbench/.vscode-server/bin/e10f2369d0d9614a452462f2e01cdc4aa9486296/bin/remote-cli:/home/workbench/.local/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/opt/ngc-cli:/home/workbench/.local/bin:/home/workbench/.vscode-server/bin/e10f2369d0d9614a452462f2e01cdc4aa9486296/bin/remote-cli:/home/workbench/.local/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/opt/ngc-cli:/home/workbench/.local/bin:/home/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/bin: home/project/Tao-5.5/tao_tutorials/no

In [20]:
!cat ~/.tao_mounts.json

{
    "Mounts": [
        {
            "source": "/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit",
            "destination": "/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit",
            "type": "bind",
            "access": "rw"
        },
        {
            "source": "/var/run/docker.sock",
            "destination": "/var/run/docker.sock"
        },
        {
            "source": "/var/host-run",
            "destination": "/var/host-run"
        }
    ],
    "DockerOptions": {
        "shm_size": "16G",
        "ulimits": {
            "memlock": -1,
            "stack": 67108864
        },
        "user": "1000:1000",
        "network": "host",
        "privileged": true
    }
}

## 1. Installing the TAO launcher <a class="anchor" id="head-1"></a>
The TAO launcher is a python package distributed as a python wheel listed in the `nvidia-pyindex` python index. You may install the launcher by executing the following cell.

Please note that TAO Toolkit recommends users to run the TAO launcher in a virtual env with python 3.6.9. You may follow the instruction in this [page](https://virtualenvwrapper.readthedocs.io/en/latest/install.html) to set up a python virtual env using the `virtualenv` and `virtualenvwrapper` packages. Once you have setup virtualenvwrapper, please set the version of python to be used in the virtual env by using the `VIRTUALENVWRAPPER_PYTHON` variable. You may do so by running

```sh
export VIRTUALENVWRAPPER_PYTHON=/path/to/bin/python3.x
```
where x >= 6 and <= 8

We recommend performing this step first and then launching the notebook from the virtual environment. In addition to installing TAO python package, please make sure of the following software requirements:
* python >=3.7, <=3.10.x
* docker-ce > 19.03.5
* docker-API 1.40
* nvidia-container-toolkit > 1.3.0-1
* nvidia-container-runtime > 3.4.0-1
* nvidia-docker2 > 2.5.0-1
* nvidia-driver > 455+

Once you have installed the pre-requisites, please log in to the docker registry nvcr.io by following the command below

```sh
docker login nvcr.io
```

You will be triggered to enter a username and password. The username is `$oauthtoken` and the password is the API key generated from `ngc.nvidia.com`. Please follow the instructions in the [NGC setup guide](https://docs.nvidia.com/ngc/ngc-overview/index.html#generating-api-key) to generate your own API key.

Please note that TAO Toolkit recommends users to run the TAO launcher in a virtual env with python >=3.6.9. You may follow the instruction in this [page](https://virtualenvwrapper.readthedocs.io/en/latest/install.html) to set up a python virtual env using the virtualenv and virtualenvwrapper packages.

In [9]:
# SKIP this step IF you have already installed the TAO launcher.
!pip3 install nvidia-pyindex
!pip3 install nvidia-tao

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... done
  Created wheel for nvidia-pyindex: filename=nvidia_pyindex-1.0.9-py3-none-any.whl size=8419 sha256=e2889afddc4fa934184173bea3715f8a9923f3456887690b4cc71a242788833e
  Stored in directory: /home/workbench/.cache/pip/wheels/2c/af/d0/7a12f82cab69f65d51107f48bcd6179e29b9a69a90546332b3
Successfully built nvidia-pyindex
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [21]:
# View the versions of the TAO launcher
!tao info

Configuration of the TAO Toolkit Instance
task_group: ['model', 'dataset', 'deploy']
format_version: 3.0
toolkit_version: 5.5.0
published_date: 08/26/2024


## 2. Prepare dataset and pre-trained model <a class="anchor" id="head-2"></a>

### 2.1 Prepare dataset

 We will be using the COCO dataset for the tutorial. The following script will download COCO dataset automatically.

In [38]:
# Create local dir
#!mkdir -p $HOST_DATA_DIR
# Download the data
#!bash $HOST_SPECS_DIR/download_coco.sh $HOST_DATA_DIR

In [22]:
# Verification
!ls -l $HOST_DATA_DIR/
!ls -l $HOST_SPECS_DIR

total 156
drwxrwxrwx 5 workbench workbench   4096 Oct  3 03:46 raw-data
drwxrwxrwx 2 workbench workbench  12288 Oct  2 19:18 test
drwxrwxrwx 2 workbench workbench 118784 Oct  2 19:18 train
drwxrwxrwx 2 workbench workbench  20480 Oct  2 19:18 valid
total 48
-rwxrwxrwx 1 workbench workbench  621 Oct  1 00:17 classmap.txt
-rwxrwxrwx 1 workbench workbench 1377 Oct  1 00:17 distill.yaml
-rwxrwxrwx 1 workbench workbench  454 Oct  3 06:28 evaluate.yaml
-rwxrwxrwx 1 workbench workbench  473 Oct  1 00:17 evaluate_distill.yaml
-rwxrwxrwx 1 workbench workbench  305 Oct  1 00:17 export.yaml
-rwxrwxrwx 1 workbench workbench  305 Oct  1 00:17 export_distill.yaml
-rwxrwxrwx 1 workbench workbench  262 Oct  1 00:17 gen_trt_engine.yaml
-rwxrwxrwx 1 workbench workbench  516 Oct  1 00:17 infer.yaml
-rwxrwxrwx 1 workbench workbench  516 Oct  1 00:17 infer_distill.yaml
-rwxrwxrwx 1 workbench workbench 1118 Oct  3 11:35 train-1.yaml
-rwxrwxrwx 1 workbench workbench 1118 Oct  3 14:20 train-Copy1.yaml
-rwxrwxr

### 2.2 Download pre-trained model

We will use NGC CLI to get the pre-trained models. For more details, go to [ngc.nvidia.com](ngc.nvidia.com) and click the SETUP on the navigation bar.

In [14]:
import os

# Set the environment variables
os.environ['LOCAL_PROJECT_DIR'] = '/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino'
os.environ['CLI'] = 'ngccli_cat_linux.zip'

# Verify the environment variables
print(f"LOCAL_PROJECT_DIR: {os.environ['LOCAL_PROJECT_DIR']}")
print(f"CLI: {os.environ['CLI']}")

# Create the ngccli directory
!mkdir -p $LOCAL_PROJECT_DIR/ngccli

# Remove any previously existing CLI installations
!rm -rf $LOCAL_PROJECT_DIR/ngccli/*
!wget "https://ngc.nvidia.com/downloads/$CLI" -P $LOCAL_PROJECT_DIR/ngccli
!unzip -u "$LOCAL_PROJECT_DIR/ngccli/$CLI" -d $LOCAL_PROJECT_DIR/ngccli/
!rm $LOCAL_PROJECT_DIR/ngccli/*.zip 

# Update the PATH environment variable
os.environ["PATH"] = f"{os.getenv('LOCAL_PROJECT_DIR')}/ngccli/ngc-cli:{os.getenv('PATH')}"
print(f"Updated PATH: {os.environ['PATH']}")

LOCAL_PROJECT_DIR: /nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino
CLI: ngccli_cat_linux.zip
--2024-10-09 01:21:33--  https://ngc.nvidia.com/downloads/ngccli_cat_linux.zip
Resolving ngc.nvidia.com (ngc.nvidia.com)... 108.157.142.78, 108.157.142.45, 108.157.142.56, ...
Connecting to ngc.nvidia.com (ngc.nvidia.com)|108.157.142.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47813875 (46M) [application/zip]
Saving to: ‘/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/ngccli/ngccli_cat_linux.zip’

ngccli_cat_linux.zi 100%[===================>]  45.60M  46.0MB/s    in 1.0s    

2024-10-09 01:21:35 (46.0 MB/s) - ‘/nvidia-workbench/brian-english1978-nim-anywhere/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/ngccli/ngccli_cat_linux.zip’ saved [47813875/47813875]

Archive:  /nvidia-workbench/brian-english1978-nim-anywhere/Tao-5

In [15]:
!ngc registry model list nvidia/tao/pretrained_dino_nvimagenet:*

NGC_API_KEY is a deprecated environment variable. Please use NGC_CLI_API_KEY instead.
+-------+-------+-------+-------+-------+-------+-------+-------+-------+
| Versi | Accur | Epoch | Batch | GPU   | Memor | File  | Statu | Creat |
| on    | acy   | s     | Size  | Model | y Foo | Size  | s     | ed    |
|       |       |       |       |       | tprin |       |       | Date  |
|       |       |       |       |       | t     |       |       |       |
+-------+-------+-------+-------+-------+-------+-------+-------+-------+
| resne |       |       | 1     | V100  | 292.9 | 292.8 | UPLOA | Jul   |
| t50   |       |       |       |       |       | 9 MB  | D_COM | 17,   |
|       |       |       |       |       |       |       | PLETE | 2023  |
| gcvit |       |       | 1     | V100  | 47.7  | 47.7  | UPLOA | Jul   |
| _xxti |       |       |       |       |       | MB    | D_COM | 17,   |
| ny_nv |       |       |       |       |       |       | PLETE | 2023  |
| image |       |       | 

In [17]:
# Pull pretrained model from NGC
!ngc registry model download-version nvidia/tao/pretrained_dino_nvimagenet:fan_small_hybrid_nvimagenet --dest $LOCAL_PROJECT_DIR

NGC_API_KEY is a deprecated environment variable. Please use NGC_CLI_API_KEY instead.
Getting files to download...
⠋ ━━ • 0… • Remaining: … • ? • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠙ ━━ • 0… • Remaining: … • ? • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠹ ━━ • 0… • Remaining: … • ? • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠼ ━━ • 0… • Remaining: … • ? • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠴ ━━ • 0… • Remaining: … • ? • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠦ ━━ • 0… • Remaining: … • ? • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠇ ━━ • 0… • Remaining: … • ? • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠏ ━━ • 0… • Remaining: … • ? • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠋ ━━ • … • Remaining: 0… • … • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠹ ━━ • … • Remaining: 0… • … • Elapsed: 0… • Total: 1 - Completed: 0 - Failed: 0
⠸ ━━ • … • Remaining: 0… • … • Elapsed: 0… • Total: 1 - Completed: 0 - Fail

In [23]:
import os

# Set the environment variable
os.environ['LOCAL_PROJECT_DIR'] = '/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit'

print("Check that model is downloaded into dir.")
!ls -l $LOCAL_PROJECT_DIR/dino/pretrained_dino_nvimagenet_vfan_small_hybrid_nvimagenet/

Check that model is downloaded into dir.
total 102280
-rwxrwxrwx 1 workbench workbench 104734139 Oct  9 01:22 fan_small_hybrid_nvimagenet.pth


## 3. Provide training specification <a class="anchor" id="head-3"></a>

We provide specification files to configure the training parameters including:

* dataset: configure the dataset and augmentation methods
    * train_data_sources:
        * image_dir: annotation file for train data. required to be in COCO json format
        * json_file: the root directory for train images
    * val_data_sources: 
        * image_dir: the root directory for validation images
        * json_file: annotation file for validation data. required to be in COCO json format
    * num_classes: number of classes of you training data
    * batch_size: batch size for dataloader
    * workers: number of workers to do data loading
* model: configure the model setting
    * pretrained_backbone_path: path to the pretrained backbone model. ResNet50, FAN-variants, and GCViT-variants are supported
    * num_feature_levels: number of feature levels used from backbone
    * dec_layers: number of decoder layers
    * enc_layers: number of encoder layers
    * num_queries: number of queries for the model
    * num_select: number of top-k proposals to select from
    * use_dn: flag to enable denoising during training
    * dropout_ratio: drop out ratio
* train: configure the training hyperparameters
    * num_gpus: number of gpus 
    * num_nodes: number of nodes (num_nodes=1 for single node)
    * val_interval: validation interval
    * optim:
        * lr_backbone: learning rate for backbone
        * lr: learning rate for the rest of the model
        * lr_steps: learning rate decay step milestone (MultiStep)
    * num_epochs: number of epochs
    * activation_checkpoint: recompute activations in the backward to save GPU memory. Default is `True`.
    * precision: If set to fp16, the training is run on Automatic Mixed Precision (AMP)
    * distributed_strategy: Default is `ddp`. `ddp_sharded` is also supported.

* **Note that the sample spec is not meant to produce SOTA accuracy on COCO. To reproduce SOTA, you should set `num_feature_levels` as 4 to match the original params. In addition, the use of NVImageNet weight also cause a slightly lower mAP when compared with ImageNet weight.**

Please refer to the TAO documentation about DINO to get all the parameters that are configurable.


In [24]:
!cat $HOST_SPECS_DIR/train.yaml

train:
  num_gpus: 2
  num_nodes: 1
  validation_interval: 1
  optim:
    lr_backbone: 2e-05
    lr: 2e-4
    lr_steps: [11]
    momentum: 0.9
  num_epochs: 50
dataset:
  train_data_sources:
    - image_dir: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data/raw-data/train/
      json_file: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data/raw-data/train/_annotations.coco.json
  val_data_sources:
    - image_dir: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data/raw-data/valid/
      json_file: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data/raw-data/valid/_annotations.coco.json
  num_classes: 124
  batch_size: 4
  workers: 8
  augmentation:
    fixed_padding: False
model:
  backbone: fan_small
  train_backbone: True
  pretrained_backbone_path: /workspace/tao-experiments/dino/pretrained_dino_nvimagenet_vfan_small_hybrid_nvimagenet/fan_small_hybrid_nvimagenet.pth
  num_feature_levels

## 4. Run TAO training <a class="anchor" id="head-4"></a>
* Provide the sample spec file and the output directory location for models
* Evaluation uses COCO metrics. For more info, please refer to: https://cocodataset.org/#detection-eval
* *WARNING*: [according to the orirginal paper](https://arxiv.org/abs/2203.03605), COCO training was conducted using 8 A100 gpus. As a result, **we highly recommend that you run training with multiple high-end gpus (e.g. V100, A100)**
* COCO per-epoch training time on a single GPU (the hours may vary depending on the data location, network speed, and etc).

<table>
  <tr>
    <th>GPU Type</th>
    <th>Time (hrs)</th>
  </tr>
  <tr>
    <td>1 x V100 32GB</td>
    <td>21.5</td>
  </tr>
  <tr>
    <td>1 x A100 80GB</td>
    <td>8</td>
  </tr>
</table>

* For this demonstration, we changed the architectures from the original implementation so that the training can be completed faster (e.g. num_queries 900 -> 300, num_feature_levels 4 -> 2, and etc).
* Unlike the [original DINO paper](https://arxiv.org/abs/2203.03605), we used more advanced backbone called [FAN](https://arxiv.org/abs/2204.12451) that has proven to achieve higher downstream results compared to ResNet, Swin, and ConvNext. 
* If you wish to speed up training, you may try to set `train.precision=fp16` for mixed precision training

In [25]:
# NOTE: The following paths are set from the perspective of the TAO Docker.

# The data is saved here
%env DATA_DIR=/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data
%env SPECS_DIR=/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/specs
%env RESULTS_DIR=/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/results

env: DATA_DIR=/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data
env: SPECS_DIR=/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/specs
env: RESULTS_DIR=/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/results


In [26]:
!echo $HOST_DATA_DIR

/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data


In [27]:
import os

# Set the environment variables
os.environ['DATA_DIR'] = '/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data'
os.environ['SPECS_DIR'] = '/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/specs'
os.environ['RESULTS_DIR'] = '/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/results'

# Verify the environment variables
print(f"DATA_DIR: {os.environ['DATA_DIR']}")
print(f"SPECS_DIR: {os.environ['SPECS_DIR']}")
print(f"RESULTS_DIR: {os.environ['RESULTS_DIR']}")

# Check if the train.yaml file exists
train_yaml_path = os.path.join(os.environ['SPECS_DIR'], 'train.yaml')
if not os.path.exists(train_yaml_path):
    raise FileNotFoundError(f"The indicated experiment spec file `{train_yaml_path}` doesn't exist!")

# Print the contents of the specs directory
print("Contents of the specs directory:")
!ls -l $SPECS_DIR

# Check file permissions
print("File permissions for train.yaml:")
!ls -l $train_yaml_path

# Verify the contents of the specs directory inside the Docker container
print("Contents of the specs directory inside the Docker container:")
!docker run --rm -v $SPECS_DIR:/specs nvcr.io/nvidia/tao/tao-toolkit:5.5.0-pyt ls -l /specs

# Check the PATH and locate the tao command within the Docker container
print("Checking PATH and locating tao command within the Docker container:")
!docker run --rm -v $SPECS_DIR:/specs nvcr.io/nvidia/tao/tao-toolkit:5.5.0-pyt /bin/bash -c "echo $PATH && find / -name tao"

# Run the TAO command with the correct volume mount and recommended flags
print("For multi-GPU, change train.num_gpus in train.yaml based on your machine")
print("For multi-node, change train.num_gpus and num_nodes in train.yaml based on your machine")
print("If you face out of memory issue, you may reduce the batch size in the spec file by passing dataset.batch_size=2")
!docker run --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --rm -v $SPECS_DIR:/specs -v $RESULTS_DIR:/results nvcr.io/nvidia/tao/tao-toolkit:5.5.0-pyt /bin/bash -c "export PATH=$PATH:/opt/nvidia/tao-toolkit && tao model dino train -e /specs/train.yaml results_dir=/results"

DATA_DIR: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/data
SPECS_DIR: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/specs
RESULTS_DIR: /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/results
Contents of the specs directory:
total 48
-rwxrwxrwx 1 workbench workbench  621 Oct  1 00:17 classmap.txt
-rwxrwxrwx 1 workbench workbench 1377 Oct  1 00:17 distill.yaml
-rwxrwxrwx 1 workbench workbench  454 Oct  3 06:28 evaluate.yaml
-rwxrwxrwx 1 workbench workbench  473 Oct  1 00:17 evaluate_distill.yaml
-rwxrwxrwx 1 workbench workbench  305 Oct  1 00:17 export.yaml
-rwxrwxrwx 1 workbench workbench  305 Oct  1 00:17 export_distill.yaml
-rwxrwxrwx 1 workbench workbench  262 Oct  1 00:17 gen_trt_engine.yaml
-rwxrwxrwx 1 workbench workbench  516 Oct  1 00:17 infer.yaml
-rwxrwxrwx 1 workbench workbench  516 Oct  1 00:17 infer_distill.yaml
-rwxrwxrwx 1 workbench workbench 1118 Oct  3 11:35 train-1.yaml
-rwxrwxrwx 1 workbench wo

File permissions for train.yaml:
-rwxrwxrwx 1 workbench workbench 1118 Oct  3 14:21 /project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/specs/train.yaml
Contents of the specs directory inside the Docker container:

=== TAO Toolkit PyTorch ===

NVIDIA Release 5.5.0-PyT (build 88113656)
TAO Toolkit Version 5.5.0

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

This container image and its contents are governed by the TAO Toolkit End User License Agreement.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/tao-toolkit-software-license-agreement

NOTE: The SHMEM allocation limit is set to the default of 64MB.  This may be
   insufficient for TAO Toolkit.  NVIDIA recommends the use of the following flags:
   docker run --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 ...

total 4
drwxr-xr-x 2 root root 4096 Oct  3 15:19 train.yaml
Checking P

tao model dino train -e /specs/train.yaml results_dir=/results

In [15]:
print('Trained checkpoints:')
print('---------------------')
!ls -ltrh $HOST_RESULTS_DIR/train

Trained checkpoints:
---------------------
ls: cannot access '/project/Tao-5.5/tao_tutorials/notebooks/tao_launcher_starter_kit/dino/results/train': No such file or directory


In [ ]:
# You can set NUM_EPOCH to the epoch corresponding to any saved checkpoint
# %env NUM_EPOCH=029

# Get the name of the checkpoint corresponding to your set epoch
# tmp=!ls $HOST_RESULTS_DIR/train/*.pth | grep epoch_$NUM_EPOCH
# %env CHECKPOINT={tmp[0]}

# Or get the latest checkpoint
os.environ["CHECKPOINT"] = os.path.join(os.getenv("HOST_RESULTS_DIR"), "train/dino_model_latest.pth")

print('Rename a trained model: ')
print('---------------------')
!cp $CHECKPOINT $HOST_RESULTS_DIR/train/dino_model.pth
!ls -ltrh $HOST_RESULTS_DIR/train/dino_model.pth

## 5. Evaluate a trained model <a class="anchor" id="head-5"></a>

In this section, we run the `evaluate` tool to evaluate the trained model and produce the mAP metric.

We provide evaluate.yaml specification files to configure the evaluate parameters including:

* model: configure the model setting
    * this config should remain same as your trained model's configuration.
* dataset: configure the dataset and augmentation methods
    * test_data_sources:
        * image_dir: the root directory for evaluatation images    
        * json_file: annotation file for evaluatation data. required to be in COCO json format.
    * num_classes: number of classes you used for training
    * eval_class_ids: classes you would like to evaluate. \
                    Note that current config file will evaluate only on class 1 (person in COCO dataset)\
                    If you remove this from config file, it will evaluate and compute the average over entire classes.
    * batch_size
    * workers
* evaluate:
    * num_gpus: number of gpus
    * conf_threshold: a threshold for confidence scores

* **NOTE: You need to change the model path in evaluate.yaml file based on your setting.**

In [14]:
# Evaluate on TAO model
!tao model dino evaluate \
            -e $SPECS_DIR/evaluate.yaml \
            evaluate.checkpoint=$RESULTS_DIR/train/dino_model.pth \
            results_dir=$RESULTS_DIR/

2024-10-05 00:46:27,559 [TAO Toolkit] [INFO] root 160: Registry: ['nvcr.io']
2024-10-05 00:46:27,665 [TAO Toolkit] [INFO] nvidia_tao_cli.components.instance_handler.local_instance 360: Running command in container: nvcr.io/nvidia/tao/tao-toolkit:5.5.0-pyt
2024-10-05 00:46:28,500 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 301: Printing tty value True
[2024-10-05 00:46:34,913 - TAO Toolkit - matplotlib.font_manager - INFO] generated new fontManager
ERROR: The indicated experiment spec file `/evaluate.yaml` doesn't exist!
2024-10-05 00:46:36,129 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 363: Stopping container.


## 6. Visualize Inferences <a class="anchor" id="head-6"></a>
In this section, we run the `inference` tool to generate inferences on the trained models and visualize the results. The `inference` tool produces annotated image outputs and txt files that contain prediction information.

We provide evaluate.yaml specification files to configure the evaluate parameters including:

* model: configure the model setting
    * this config should remain same as your trained model's configuration
* dataset: configure the dataset and augmentation methods
    * infer_data_sources:
        * image_dir: the list of directories for inference images
        * classmap: 
    * num_classes: number of classes you used for training
    * batch_size
    * workers
* inference
    * conf_threshold: the confidence score threshold
    * color_map: the color mapping for each class. The predicted bbox will be drawn with mapped color for each class
* **NOTE: You need to change the model path in infer.yaml file based on your setting.**

In [ ]:
# copy classmap to annotation directory
!cp $HOST_SPECS_DIR/classmap.txt $HOST_DATA_DIR/raw-data/annotations/

In [ ]:
!tao model dino inference \
        -e $SPECS_DIR/infer.yaml \
        inference.checkpoint=$RESULTS_DIR/train/dino_model.pth \
        results_dir=$RESULTS_DIR/

In [ ]:
# Install matplotlib
!pip3 install "matplotlib>=3.3.3, <4.0"

# Import necessary libraries
import matplotlib.pyplot as plt
import os
from math import ceil

# Define valid image extensions
valid_image_ext = ['.jpg']

# Define the function to visualize images
def visualize_images(output_path, num_cols=4, num_images=10):
    num_rows = int(ceil(float(num_images) / float(num_cols)))
    f, axarr = plt.subplots(num_rows, num_cols, figsize=[80,30])
    f.tight_layout()
    a = [os.path.join(output_path, image) for image in os.listdir(output_path) 
         if os.path.splitext(image)[1].lower() in valid_image_ext]
    for idx, img_path in enumerate(a[:num_images]):
        col_id = idx % num_cols
        row_id = idx // num_cols
        img = plt.imread(img_path)
        axarr[row_id, col_id].imshow(img)

In [ ]:
# Visualizing the sample images.
IMAGE_DIR = os.path.join(os.environ['HOST_RESULTS_DIR'], "inference", "images_annotated")
COLS = 2 # number of columns in the visualizer grid.
IMAGES = 4 # number of images to visualize.

visualize_images(IMAGE_DIR, num_cols=COLS, num_images=IMAGES)

## 7. Deploy <a class="anchor" id="head-7"></a>

In [ ]:
# Export the RGB model to ONNX model
!tao model dino export \
           -e $SPECS_DIR/export.yaml \
           export.checkpoint=$RESULTS_DIR/train/dino_model.pth \
           export.onnx_file=$RESULTS_DIR/export/dino_model.onnx \
           results_dir=$RESULTS_DIR/

In [ ]:
# Generate TensorRT engine using tao deploy
!tao deploy dino gen_trt_engine -e $SPECS_DIR/gen_trt_engine.yaml \
                               gen_trt_engine.onnx_file=$RESULTS_DIR/export/dino_model.onnx \
                               gen_trt_engine.trt_engine=$RESULTS_DIR/gen_trt_engine/dino_model.engine \
                               results_dir=$RESULTS_DIR

In [ ]:
# Evaluate with generated TensorRT engine
!tao deploy dino evaluate -e $SPECS_DIR/evaluate.yaml \
                              evaluate.trt_engine=$RESULTS_DIR/gen_trt_engine/dino_model.engine \
                              results_dir=$RESULTS_DIR/

In [ ]:
# Inference with generated TensorRT engine
!tao deploy dino inference -e $SPECS_DIR/infer.yaml \
                              inference.trt_engine=$RESULTS_DIR/gen_trt_engine/dino_model.engine \
                              results_dir=$RESULTS_DIR/

In [ ]:
# Visualizing the sample images.
IMAGE_DIR = os.path.join(os.environ['HOST_RESULTS_DIR'], "trt_inference", "images_annotated")
COLS = 2 # number of columns in the visualizer grid.
IMAGES = 4 # number of images to visualize.

visualize_images(IMAGE_DIR, num_cols=COLS, num_images=IMAGES)

This notebook has come to an end.